In [1]:
import numpy as np
import os
import hnswlib
from sklearn.neighbors import NearestNeighbors
import time

In [2]:
METRIC = 'angular'  # 'angular' (for cosine) or 'euclidean' (for L2)
K = 10              # Number of neighbors to find
N_QUERIES = 1_000   # How many vectors to test (randomly sampled)
FEATURE_DIM = 768  # The dimension of your vectors (e.g., 2048)

# --- Set your file paths here ---
FEATURES_FILE = 'master_features_DINO_yolo_pose_multicentroid.npy'  # <-- Path to your feature vectors (NumPy .npy file)
HNSWLIB_INDEX_PATH = 'sweater_hnsw_DINO_yolo_pose_multicentroid.bin'     # <-- Path to your HNSWLib index



In [3]:
# --- Choose which index to test ---
# Set this to 'annoy', 'faiss-hnsw', 'faiss-flatip', 'faiss-flatl2', 'hnswlib', or 'voyager'
INDEX_TO_TEST = 'hnswlib'
# --- ------------------------------

In [4]:
print(f"Starting recall calculation for: {INDEX_TO_TEST.upper()}")
print(f"Metric: {METRIC} | K: {K} | Queries: {N_QUERIES}\n")


# ---
# 2. LOAD & PREPARE DATA
# ---
print(f"Loading data from {FEATURES_FILE}...")
if not os.path.exists(FEATURES_FILE):
    print(f"Error: Data file not found at {FEATURES_FILE}")
    # In a notebook, you might want to 'raise' instead of 'exit'
    # raise FileNotFoundError(f"Data file not found at {FEATURES_FILE}")
else:
    data = np.load(FEATURES_FILE).astype('float32')
    n_total, dim = data.shape
    print(f"Data loaded: {n_total} vectors, {dim} dimensions")

    # --- CRITICAL: NORMALIZE DATA for ANGULAR/COSINE METRIC ---
    if METRIC == 'angular':
        print("Normalizing data for angular (cosine) metric...")
        # Add a small epsilon to prevent division by zero for zero-vectors
        epsilon = 1e-12
        norms = np.linalg.norm(data, axis=1, keepdims=True)
        data = data / (norms + epsilon)
        print("Data normalized.")

Starting recall calculation for: HNSWLIB
Metric: angular | K: 10 | Queries: 1000

Loading data from master_features_DINO_yolo_pose_multicentroid.npy...
Data loaded: 35999 vectors, 768 dimensions
Normalizing data for angular (cosine) metric...
Data normalized.


In [5]:
# ---
# 3. SAMPLE QUERIES
# ---
print(f"Sampling {N_QUERIES} query vectors...")
query_indices = np.random.choice(n_total, N_QUERIES, replace=False)
queries = data[query_indices]


Sampling 1000 query vectors...


In [6]:
# ---
# 4. CALCULATE GROUND TRUTH (The "Correct" Answers)
# ---
print("Calculating ground truth with sklearn.NearestNeighbors...")
start_time = time.perf_counter()

# Use 'cosine' metric for sklearn, which is equivalent to 'angular'
sklearn_metric = 'cosine' if METRIC == 'angular' else 'euclidean'
nn_exact = NearestNeighbors(n_neighbors=K, algorithm='brute', metric=sklearn_metric)
nn_exact.fit(data)

# Get the indices of the true nearest neighbors
ground_truth_indices = nn_exact.kneighbors(queries, return_distance=False)

end_time = time.perf_counter()
print(f"Ground truth calculation complete. (Took {end_time - start_time:.2f}s)")

Calculating ground truth with sklearn.NearestNeighbors...
Ground truth calculation complete. (Took 0.50s)


In [7]:
# ---
# 5. LOAD YOUR PRE-BUILT INDEX
# ---
print(f"\nLoading index: {INDEX_TO_TEST.upper()}...")
ann_indices = []
start_time = time.perf_counter()

if INDEX_TO_TEST == 'annoy':
    index_path = ANNOY_INDEX_PATH
    if not os.path.exists(ANNOY_INDEX_PATH):
        print(f"Error: Annoy index file not found at {ANNOY_INDEX_PATH}")
        # raise FileNotFoundError(f"Annoy index file not found at {ANNOY_INDEX_PATH}")
    else:
        ann_index = AnnoyIndex(FEATURE_DIM, METRIC)
        ann_index.load(ANNOY_INDEX_PATH)
        print(f"Annoy index loaded from {ANNOY_INDEX_PATH}")
        
        # Query Annoy one by one
        for vec in queries:
            # search_k = -1 means use default (n_trees * k)
            indices = ann_index.get_nns_by_vector(vec, K, search_k=-1)
            ann_indices.append(indices)
        ann_indices = np.array(ann_indices)
        
elif INDEX_TO_TEST == 'faiss-hnsw':
        index_path = FAISS_HNSW_INDEX_PATH
        if not os.path.exists(FAISS_HNSW_INDEX_PATH):
            raise FileNotFoundError(f"Faiss HNSW index file not found at {FAISS_HNSW_INDEX_PATH}")
        
        ann_index = faiss.read_index(FAISS_HNSW_INDEX_PATH)
        print(f"Faiss HNSW index loaded from {FAISS_HNSW_INDEX_PATH}")
        # Set efSearch for HNSW
        try:
            ann_index.hnsw.efSearch = 64
            print(f"Set Faiss HNSW efSearch = {ann_index.hnsw.efSearch}")
        except AttributeError:
            print("Warning: This index does not appear to be a Faiss HNSW index.")
        
        _, ann_indices = ann_index.search(queries, K)
    
elif INDEX_TO_TEST == 'faiss-flatip':
        index_path = FAISS_FLATIP_INDEX_PATH
        if not os.path.exists(FAISS_FLATIP_INDEX_PATH):
            raise FileNotFoundError(f"Faiss FlatIP index file not found at {FAISS_FLATIP_INDEX_PATH}")
        
        ann_index = faiss.read_index(FAISS_FLATIP_INDEX_PATH)
        print(f"Faiss FlatIP index loaded from {FAISS_FLATIP_INDEX_PATH}")
        # No search parameters needed for Flat indexes
        _, ann_indices = ann_index.search(queries, K)

elif INDEX_TO_TEST == 'faiss-flatl2':
        index_path = FAISS_FLATL2_INDEX_PATH
        if not os.path.exists(FAISS_FLATL2_INDEX_PATH):
            raise FileNotFoundError(f"Faiss FlatL2 index file not found at {FAISS_FLATL2_INDEX_PATH}")
        
        ann_index = faiss.read_index(FAISS_FLATL2_INDEX_PATH)
        print(f"Faiss FlatL2 index loaded from {FAISS_FLATL2_INDEX_PATH}")
        # No search parameters needed for Flat indexes
        _, ann_indices = ann_index.search(queries, K)

elif INDEX_TO_TEST == 'hnswlib':
        index_path = HNSWLIB_INDEX_PATH
        if hnswlib is None:
            raise ImportError("hnswlib library is not installed.")
        if not os.path.exists(HNSWLIB_INDEX_PATH):
            raise FileNotFoundError(f"HNSWLib index file not found at {HNSWLIB_INDEX_PATH}")
        
        hnsw_space = 'ip' if METRIC == 'angular' else 'l2'
        ann_index = hnswlib.Index(space=hnsw_space, dim=FEATURE_DIM)
        # We must tell load_index the max_elements
        ann_index.load_index(HNSWLIB_INDEX_PATH, max_elements=n_total)
        print(f"HNSWLib index loaded from {HNSWLIB_INDEX_PATH}")
        
        # Set search-time parameter
        ann_index.set_ef(64)
        print(f"Set HNSWLib efSearch = 64")
        
        # Query HNSWLib
        indices, _ = ann_index.knn_query(queries, k=K)
        ann_indices = indices

elif INDEX_TO_TEST == 'voyager':
        index_path = VOYAGER_INDEX_PATH
        if voyager is None:
            raise ImportError("voyager library is not installed.")
        if not os.path.exists(VOYAGER_INDEX_PATH):
            raise FileNotFoundError(f"Voyager index file not found at {VOYAGER_INDEX_PATH}")
        
        ann_index = voyager.Index.load(VOYAGER_INDEX_PATH)
        print(f"Voyager index loaded from {VOYAGER_INDEX_PATH}")
        
        # Set search-time parameter
        ann_index.ef = 64
        print(f"Set Voyager efSearch = 64")
        
        # Query Voyager
        indices, _ = ann_index.query(queries, k=K)
        ann_indices = indices
        
else:
    print(f"Error: Unknown INDEX_TO_TEST value: '{INDEX_TO_TEST}'")
    # raise ValueError(f"Unknown INDEX_TO_TEST value: '{INDEX_TO_TEST}'")

end_time = time.perf_counter()
if len(ann_indices) > 0:
    print(f"ANN query complete. (Took {end_time - start_time:.2f}s)")




Loading index: HNSWLIB...
HNSWLib index loaded from sweater_hnsw_DINO_yolo_pose_multicentroid.bin
Set HNSWLib efSearch = 64
ANN query complete. (Took 0.27s)


In [8]:
# ---
# 6. CALCULATE RECALL
# ---
def calculate_recall_score(ground_truth, ann_results, k):
    """Calculates Recall@K."""
    if len(ann_results) == 0:
        return 0.0
        
    n_queries, _ = ground_truth.shape
    total_correct = 0
    
    for i in range(n_queries):
        # Find the intersection (common neighbors)
        correct_for_query = len(set(ground_truth[i]) & set(ann_results[i]))
        total_correct += correct_for_query
        
    # Recall = total correct items / (total items we tried to find)
    return total_correct / (n_queries * k)

if len(ann_indices) > 0:
    recall_score = calculate_recall_score(ground_truth_indices, ann_indices, K)
    print("\n---")
    print(f"Final Result for {INDEX_TO_TEST.upper()}:")
    print(f"Recall@{K}: {recall_score:.4f}  ({recall_score * 100:.2f}%)")
    print("---")
else:
    print("\nRecall calculation skipped due to errors.")


---
Final Result for HNSWLIB:
Recall@10: 0.9891  (98.91%)
---


In [9]:
index_size = os.path.getsize(index_path) / (1024 * 1024)
print(f" Index size on disk: {index_size:.2f} MB")

 Index size on disk: 110.57 MB
